# Báo cáo: Giải Hệ Phương Trình và Phân Tích Hiệu Năng (Phần 3)

Trong phần này, chúng ta sẽ thực hiện giải hệ phương trình tuyến tính $A\mathbf{x} = \mathbf{b}$ bằng 3 phương pháp khác nhau:
1. **Khử Gauss (Gauss Elimination)**: Phương pháp giải trực tiếp với *partial pivoting*.
2. **Phân rã Cholesky**: Phương pháp giải trực tiếp dành cho ma trận đối xứng xác định dương (SPD).
3. **Lặp Gauss-Seidel**: Phương pháp giải lặp, hội tụ khi ma trận chéo trội chặt hoặc SPD.

Các tiêu chí phân tích bao gồm thời gian chạy, sai số tương đối và độ ổn định của từng thuật toán.

## 1. Import Thư Viện và Các Hàm Solver
Chúng ta import các hàm từ file `solvers.py` và `benchmark.py` đã triển khai.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hilbert

# Import từ file trong project
from solvers import solve_gauss, solve_cholesky, solve_gauss_seidel, condition_number
from benchmark import timeit, relative_residual, generate_random_spd, run_time_benchmark, print_summary_table

## 2. Thực Nghiệm Thời Gian và Độ Phức Tạp (Benchmark)
Chúng ta sẽ thực hiện benchmark với kích thước ma trận $n \in \{50, 100, 200, 500, 1000\}$.
Ma trận thử nghiệm là ma trận ngẫu nhiên SPD (đảm bảo cả 3 phương pháp đều có thể giải được).

In [2]:
sizes = [50, 100, 200, 500, 1000]
results = run_time_benchmark(sizes=sizes, n_runs=5)
print_summary_table(results)

### Biểu đồ Thời gian chạy vs Kích thước n (Log-Log)
Biểu đồ so sánh thời gian của các thuật toán với đường tham chiếu độ phức tạp lý thuyết $\mathcal{O}(n^3)$. Gauss-Seidel do tính hội tụ cực nhanh trên ma trận chéo trội SPD nên tốn ít thời gian nhất, trong khi Gauss và Cholesky có độ dốc tương đồng với $\mathcal{O}(n^3)$.

<p align="center">
  <img src="benchmark_loglog.png" alt="Benchmark Log-log" width="800"/>
</p>

### Nhận xét Benchmark
Từ bảng kết quả thực nghiệm với các kích thước ma trận $n \in \{50, 100, 200, 500, 1000\}$, ta có các nhận xét sau:

1. **Thời gian chạy và độ phức tạp:**
   - **Khử Gauss**: Nhờ việc **vector hóa** vòng lặp bên trong thông qua NumPy, thời gian chạy rất tối ưu. Tại $n=1000$, phương pháp này chỉ mất khoảng **~1.1 đến 1.6 giây**. 
   - **Gauss-Seidel**: Do ma trận ngẫu nhiên SPD được tạo ra có tính chéo trội rất mạnh, Gauss-Seidel hội tụ cực kỳ nhanh (chỉ mất ~23 lần lặp cho $n=1000$). Thời gian thực thi là **nhanh nhất**, chỉ tốn khoảng **~0.17 giây**.
   - **Cholesky**: Thuật toán do cài đặt bằng cấu trúc vòng lặp thuần Python (chưa vector hóa) nên thời gian tăng đáng kể. Tại $n=1000$, nó mất khoảng **~21 giây**, chậm hơn so với Gauss nhưng vẫn tuân theo tiệm cận $\mathcal{O}(n^3)$ trên đồ thị log-log.

2. **Độ chính xác (Sai số tương đối):**
   - **Khử Gauss và Cholesky**: Đạt độ chính xác toán học rất cao, sai số luôn nằm ở ngưỡng $\sim 10^{-16}$ (machine epsilon) với mọi kích thước.
   - **Gauss-Seidel**: Dừng lặp khi sai số đạt mức dung sai `tol=1e-10`, nên sai số thực nghiệm luôn xấp xỉ $\sim 10^{-11}$, hoàn toàn chính xác theo đúng yêu cầu cài đặt.

## 3. Phân Tích Ổn Định Số
Mục đích: Quan sát cách sai số bị khuếch đại khi hệ phương trình có *số điều kiện lớn*.
- **Ma trận Hilbert ($H_n$)**: Rất nhạy với nhiễu (ill-conditioned).
- **Ma trận ngẫu nhiên SPD**: Thường có tính điều kiện tốt hơn (well-conditioned).

In [4]:
stability_sizes = [3, 5, 7, 10, 12, 15]

hilbert_data = {'sizes': stability_sizes, 'cond': [], 'err_gauss': [], 'err_cholesky': []}
spd_data = {'sizes': stability_sizes, 'cond': [], 'err_gauss': [], 'err_cholesky': []}

for n in stability_sizes:
    np.random.seed(42)
    x_true = np.ones(n)
    
    # --- Hilbert ---
    H = hilbert(n)
    b_h = H @ x_true
    kappa_h = condition_number(H)
    hilbert_data['cond'].append(kappa_h)
    
    # Giải bằng Gauss
    try:
        x_g_h = solve_gauss(H, b_h)
        hilbert_data['err_gauss'].append(np.linalg.norm(x_g_h - x_true) / np.linalg.norm(x_true))
    except:
        hilbert_data['err_gauss'].append(float('nan'))
        
    # Giải bằng Cholesky
    try:
        x_c_h = solve_cholesky(H, b_h)
        hilbert_data['err_cholesky'].append(np.linalg.norm(x_c_h - x_true) / np.linalg.norm(x_true))
    except:
        hilbert_data['err_cholesky'].append(float('nan'))

    # --- Random SPD ---
    A_spd = generate_random_spd(n)
    b_spd = A_spd @ x_true
    kappa_s = condition_number(A_spd)
    spd_data['cond'].append(kappa_s)
    
    try:
        x_g_s = solve_gauss(A_spd, b_spd)
        spd_data['err_gauss'].append(np.linalg.norm(x_g_s - x_true) / np.linalg.norm(x_true))
    except:
        spd_data['err_gauss'].append(float('nan'))
        
    try:
        x_c_s = solve_cholesky(A_spd, b_spd)
        spd_data['err_cholesky'].append(np.linalg.norm(x_c_s - x_true) / np.linalg.norm(x_true))
    except:
        spd_data['err_cholesky'].append(float('nan'))


### Kết quả Biểu đồ Ổn định số

<p align="center">
  <img src="stability_analysis.png" alt="Stability Analysis" width="1000"/>
</p>

### Nhận xét Phân Tích Ổn Định Số và Kết Luận Cuối Cùng

**1. Phân Tích Ma Trận Hilbert ($H_n$) - Hệ điều kiện kém (Ill-conditioned):**
   - **Số điều kiện $\kappa(A)$**: Tăng cực nhanh theo hàm mũ. Với $n=10$, $\kappa \approx 1.6 \times 10^{13}$. Tại $n=12$, $\kappa$ tiến tới vô cực (`inf`) vượt qua giới hạn của máy tính.
   - **Độ ổn định của nghiệm**: Tại $n=10$, sai số nghiệm của Gauss và Cholesky đã lên tới $\sim 10^{-4}$. Khi $n=12$, Gauss trả về `NaN` còn Cholesky sai số cực kỳ lớn ($\sim 0.41$). Điều này minh họa thực tế rằng: với ma trận Hilbert, sai số làm tròn cực nhỏ từ dữ liệu ban đầu bị **khuếch đại hàng tỷ lần**, làm vỡ hệ thống.

**2. Phân Tích Ma Trận ngẫu nhiên SPD - Hệ điều kiện tốt (Well-conditioned):**
   - Số điều kiện $\kappa(A)$ luôn duy trì ở mức cực kỳ ổn định ($\sim 3.0$ đến $4.4$) bất chấp kích thước $n$ tăng dần.
   - Sai số tương đối của cả phương pháp Gauss và Cholesky luôn ổn định ở mức sát giới hạn máy tính ($\sim 10^{-16}$). Hệ thống giải vô cùng trơn tru và chính xác.

**🌟 KẾT LUẬN CUỐI CÙNG:**
Thông qua toàn bộ các thực nghiệm, chúng ta rút ra các kết luận quan trọng sau:
1. **Hiệu năng lập trình rất quan trọng**: Dù thuật toán có cùng độ phức tạp toán học $\mathcal{O}(n^3)$, nhưng khi thực thi thực tế, việc sử dụng các phép toán Vector hóa (như Gauss dùng NumPy) cho tốc độ vượt trội hơn hẳn so với việc tính toán tuần tự bằng các vòng lặp lồng nhau (như Cholesky thuần Python).
2. **Thuật toán đúng là chưa đủ**: Trong máy tính, sai số làm tròn số thực (Floating point) là không thể tránh khỏi. Nếu hệ phương trình rơi vào trường hợp số điều kiện lớn (như ma trận Hilbert), thì một thuật toán hoàn hảo về mặt lý thuyết vẫn sẽ trả về kết quả sai lệch hoàn toàn.
3. **Tầm quan trọng của phương pháp Lặp**: Phương pháp Gauss-Seidel có ưu thế cực lớn với các hệ ma trận đặc biệt, đặc biệt là hệ chéo trội. Nó giúp ta tính ra nghiệm hội tụ trong khoảng thời gian cực kỳ nhỏ, tiết kiệm chi phí hơn hẳn phương pháp giải trực tiếp.